In [ ]:
# ==========================================================
# 🔧 Clean environment for DFDC + GradCAM + Blockchain (Colab T4 GPU)
# ==========================================================
!nvidia-smi

!pip install -q --upgrade pip
!pip install -q \
  torchcam>=0.4.0 \
  web3>=6.14.0 \
  opencv-python>=4.10.0 \
  pandas>=2.2.0 \
  matplotlib>=3.9.0 \
  tqdm scikit-learn


In [ ]:
import torch, numpy as np, matplotlib
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__)
print("NumPy:", np.__version__)
print("Matplotlib:", matplotlib.__version__)

In [ ]:
# ============================================================
# DFDC END-TO-END (Drive -> unzip -> organize -> frames -> train)
# Source in Drive: /content/drive/MyDrive/DFDC/{metadata.json, deepfake-detection-challenge.zip}
# Output checkpoint: /content/best_effnetb0_frames.pth
# ============================================================

# --- deps ---
!pip -q install torch torchvision opencv-python tqdm matplotlib torchcam

import os, json, random, cv2, warnings, shutil
from glob import glob
from pathlib import Path
from tqdm import tqdm

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from google.colab import drive

# ---------------- Config (adjust as needed) ----------------
DRIVE_DIR        = "/content/drive/MyDrive/DFDC"
WORK_DIR         = "/content/dfdc"           # unzip and metadata here
DATA_DIR         = "/content/data"           # organized videos (real/fake)
FRAMES_DIR       = "/content/frames"         # extracted frames (real/fake)

MAX_PER_CLASS    = 300                       # videos per class to copy (balance)
K_FRAMES_PER_VID = 8                         # frames extracted per video
IMG_SIZE         = 224
VAL_SPLIT        = 0.2
BATCH_SIZE       = 16
EPOCHS           = 20                        # train longer for higher accuracy
LR               = 1e-4
WEIGHT_DECAY     = 1e-4
NUM_WORKERS      = 2
SEED             = 42
DEVICE           = "cuda" if torch.cuda.is_available() else "cpu"
warnings.filterwarnings("ignore", category=FutureWarning)

random.seed(SEED); torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.backends.cudnn.benchmark = True
    try: torch.set_float32_matmul_precision("high")
    except: pass

print(f"✅ Using device: {DEVICE}")

In [ ]:

# ---------------- Step 0: Mount Drive & stage files ----------------
drive.mount('/content/drive')

Path(WORK_DIR).mkdir(parents=True, exist_ok=True)
src_zip  = f"{DRIVE_DIR}/deepfake-detection-challenge.zip"
src_meta = f"{DRIVE_DIR}/metadata.json"

assert os.path.exists(src_zip),  f"❌ Not found: {src_zip}"
assert os.path.exists(src_meta), f"❌ Not found: {src_meta}"

print("📂 Copying DFDC zip & metadata to working dir...")
shutil.copy(src_zip,  f"{WORK_DIR}/deepfake-detection-challenge.zip")
shutil.copy(src_meta, f"{WORK_DIR}/metadata.json")

# ---------------- Step 1: Unzip DFDC (if not already) ----------------
if not os.path.isdir(f"{WORK_DIR}/train_sample_videos"):
    print("📦 Extracting deepfake-detection-challenge.zip ...")
    !unzip -q /content/dfdc/deepfake-detection-challenge.zip -d /content/dfdc
else:
    print("✅ train_sample_videos already present")

# ---------------- Step 2: Validate metadata vs videos ----------------
meta_path = f"{WORK_DIR}/metadata.json"
vid_dir   = f"{WORK_DIR}/train_sample_videos"

assert os.path.exists(meta_path), "❌ metadata.json missing in WORK_DIR"
assert os.path.isdir(vid_dir),    "❌ train_sample_videos folder missing"

with open(meta_path, "r") as f:
    meta = json.load(f)

all_vids = [f for f in os.listdir(vid_dir) if f.lower().endswith((".mp4",".mov",".avi",".mkv"))]
overlap  = [v for v in all_vids if v in meta]

print(f"📊 metadata entries: {len(meta)}")
print(f"📹 videos found:     {len(all_vids)}")
print(f"🔗 overlap (match):  {len(overlap)}")
assert len(overlap) > 0, "❌ No filename overlap between videos and metadata.json"

In [ ]:
# ---------------- Step 3: Build a balanced subset into /content/data ----------------
for c in ("real","fake"):
    Path(f"{DATA_DIR}/{c}").mkdir(parents=True, exist_ok=True)

real_list = [v for v in overlap if meta[v]["label"].upper() == "REAL"]
fake_list = [v for v in overlap if meta[v]["label"].upper() == "FAKE"]

random.shuffle(real_list); random.shuffle(fake_list)

n_real = min(len(real_list), MAX_PER_CLASS)
n_fake = min(len(fake_list), MAX_PER_CLASS)
real_sel = real_list[:n_real]
fake_sel = fake_list[:n_fake]
print(f"🧰 Selected: REAL={len(real_sel)}  FAKE={len(fake_sel)}")

# clear old copies if any
for c in ("real","fake"):
    for f in glob(f"{DATA_DIR}/{c}/*"):
        os.remove(f)

print("📥 Copying selected videos to /content/data ...")
for fname in tqdm(real_sel, desc="Copy REAL"):
    shutil.copy(f"{vid_dir}/{fname}", f"{DATA_DIR}/real/{fname}")
for fname in tqdm(fake_sel, desc="Copy FAKE"):
    shutil.copy(f"{vid_dir}/{fname}", f"{DATA_DIR}/fake/{fname}")

print("✅ Video subset ready:",
      len(glob(f"{DATA_DIR}/real/*")), "real,",
      len(glob(f"{DATA_DIR}/fake/*")), "fake")

In [ ]:

# ---------------- Step 4: Extract K frames per video -> /content/frames ----------------
def extract_frames(src_dir, dst_dir, k=K_FRAMES_PER_VID, size=IMG_SIZE):
    Path(dst_dir).mkdir(parents=True, exist_ok=True)
    vids = []
    for ext in ("*.mp4","*.mov","*.avi","*.mkv"):
        vids += glob(os.path.join(src_dir, ext))
    print(f"[extract] {Path(src_dir).name}: {len(vids)} videos")
    for vp in tqdm(vids, desc=f"Extracting {Path(src_dir).name}"):
        vname = Path(vp).stem
        existing = glob(os.path.join(dst_dir, f"{vname}_*.jpg"))
        if len(existing) >= k:    # skip if already extracted
            continue
        cap = cv2.VideoCapture(vp)
        n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
        if n <= 0:
            cap.release(); continue
        idxs = [max(0, int((i+1)*n/(k+1))-1) for i in range(k)]
        for i, idx in enumerate(idxs):
            cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
            ok, f = cap.read()
            if not ok or f is None: continue
            f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
            h,w,_ = f.shape; s=min(h,w); y0=(h-s)//2; x0=(w-s)//2
            f = f[y0:y0+s, x0:x0+s]
            f = cv2.resize(f, (size, size))
            cv2.imwrite(os.path.join(dst_dir, f"{vname}_{i:02d}.jpg"),
                        cv2.cvtColor(f, cv2.COLOR_RGB2BGR))
        cap.release()

# fresh frames dir
for c in ("real","fake"):
    Path(f"{FRAMES_DIR}/{c}").mkdir(parents=True, exist_ok=True)

extract_frames(f"{DATA_DIR}/real", f"{FRAMES_DIR}/real")
extract_frames(f"{DATA_DIR}/fake", f"{FRAMES_DIR}/fake")

n_real_frames = len(glob(f"{FRAMES_DIR}/real/*.jpg"))
n_fake_frames = len(glob(f"{FRAMES_DIR}/fake/*.jpg"))
print(f"✅ Frames extracted: real={n_real_frames}, fake={n_fake_frames}")
assert n_real_frames>0 and n_fake_frames>0, "❌ No frames extracted — check videos."

# ---------------- Step 5: Image dataset & loaders ----------------
tfm_train = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.3,0.3,0.3,0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])
tfm_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

dataset = datasets.ImageFolder(FRAMES_DIR, transform=tfm_train)
val_size = int(VAL_SPLIT*len(dataset))
train_size = len(dataset) - val_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

# set transforms per split
train_ds.dataset.transform = tfm_train
val_ds.dataset.transform   = tfm_val

dl_train = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=True)
dl_val   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=True)

print(f"🖼️ Train imgs: {len(train_ds)} | Val imgs: {len(val_ds)}")


In [ ]:

# ---------------- Step 6: Model (EfficientNet-B0) ----------------
model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
# fine-tune last feature blocks + classifier
for n,p in model.named_parameters():
    if not any(k in n for k in ["features.6","features.7","classifier"]):
        p.requires_grad = False
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
model.to(DEVICE)

opt   = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                          lr=LR, weight_decay=WEIGHT_DECAY)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
scaler= torch.amp.GradScaler(enabled=(DEVICE=="cuda"))

def acc(o,y): return (o.argmax(1)==y).float().mean().item()

best=0.0
ckpt="/content/best_effnetb0_frames.pth"

# ---------------- Step 7: Train ----------------
for ep in range(1, EPOCHS+1):
    model.train(); tl=ta=n=0
    for x,y in tqdm(dl_train, desc=f"Epoch {ep}/{EPOCHS} [train]"):
        x,y = x.to(DEVICE), y.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda', enabled=(DEVICE=='cuda')):
            o = model(x)
            L = F.cross_entropy(o, y)
        scaler.scale(L).backward()
        scaler.step(opt)
        scaler.update()
        b = y.size(0); tl += L.item()*b; ta += acc(o,y)*b; n += b
    print(f"Train loss={tl/n:.4f} acc={ta/n:.4f}")

    model.eval(); vl=va=m=0
    with torch.no_grad():
        for x,y in tqdm(dl_val, desc=f"Epoch {ep}/{EPOCHS} [val]"):
            x,y = x.to(DEVICE), y.to(DEVICE)
            with torch.amp.autocast('cuda', enabled=(DEVICE=='cuda')):
                o = model(x)
                L = F.cross_entropy(o, y)
            b = y.size(0); vl += L.item()*b; va += acc(o,y)*b; m += b
    vl/=m; va/=m; sched.step()
    print(f"Val  loss={vl:.4f} acc={va:.4f}")

    if va > best:
        best = va
        torch.save({"model": model.state_dict(), "acc": best, "epoch": ep}, ckpt)
        print(f"✓ Saved {ckpt} (val_acc={best:.4f})")

print("🎯 Training complete. Best val acc:", best)
print("✅ Best model at:", ckpt)


In [ ]:
# ============================================================
# EVAL (UPDATED): Confusion Matrix, ROC/AUC, Classification Report, Robust Grad-CAM
# Works with frames dataset at /content/frames/{fake,real}
# Loads checkpoint: /content/best_effnetb0_frames.pth
# ============================================================

!pip -q install scikit-learn torchcam matplotlib

import warnings, os, random, numpy as np, matplotlib.pyplot as plt
from glob import glob
from pathlib import Path
from PIL import Image

import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import transforms, datasets
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

from sklearn.metrics import confusion_matrix, classification_report, roc_curve, roc_auc_score, auc

from torchcam.methods import SmoothGradCAMpp
from torchcam.utils import overlay_mask

# ---------------- Config (match your training) ----------------
FRAMES_DIR   = "/content/frames"                   # frames/{fake,real}
CKPT_PATH    = "/content/best_effnetb0_frames.pth" # your saved checkpoint
IMG_SIZE     = 224
BATCH_SIZE   = 32
NUM_WORKERS  = 2
VAL_SPLIT    = 0.2
SEED         = 42
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

warnings.filterwarnings("ignore", category=FutureWarning)
random.seed(SEED); torch.manual_seed(SEED)

assert os.path.isdir(FRAMES_DIR), f"Frames folder not found: {FRAMES_DIR}"
assert os.path.exists(CKPT_PATH), f"Checkpoint not found: {CKPT_PATH}"

# ---------------- Transforms (validation) ----------------
tfm_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
])

# ---------------- Dataset & deterministic split ----------------
full_ds = datasets.ImageFolder(FRAMES_DIR, transform=tfm_val)
val_size = int(VAL_SPLIT * len(full_ds))
train_size = len(full_ds) - val_size
g = torch.Generator().manual_seed(SEED)
train_ds, val_ds = random_split(full_ds, [train_size, val_size], generator=g)

dl_val = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=NUM_WORKERS, pin_memory=True)

idx_to_class = {v:k for k,v in full_ds.class_to_idx.items()}  # e.g., {0:'fake', 1:'real'}
print("Classes:", idx_to_class)
print(f"Validation images: {len(val_ds)}")

# ---------------- Model: EfficientNet-B0 & load weights ----------------
model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)
state = torch.load(CKPT_PATH, map_location=DEVICE)
model.load_state_dict(state["model"])
model.to(DEVICE).eval()
print(f"Loaded checkpoint acc={state.get('acc','N/A')} epoch={state.get('epoch','N/A')}")

# ---------------- Predictions on validation set ----------------
y_true, y_pred, y_prob = [], [], []
with torch.no_grad():
    for x, y in dl_val:
        x = x.to(DEVICE)
        logits = model(x)
        probs = F.softmax(logits, dim=1)
        y_true.extend(y.cpu().numpy().tolist())
        y_pred.extend(probs.argmax(1).cpu().numpy().tolist())
        y_prob.extend(probs[:, 1].cpu().numpy().tolist())  # probability of class index 1

y_true = np.array(y_true); y_pred = np.array(y_pred); y_prob = np.array(y_prob)

# ---------------- Metrics ----------------
cm = confusion_matrix(y_true, y_pred)
cm_norm = confusion_matrix(y_true, y_pred, normalize='true')
report = classification_report(y_true, y_pred, target_names=[idx_to_class[0], idx_to_class[1]])
auc_val = roc_auc_score(y_true, y_prob)

print("\nClassification Report:\n", report)
print(f"AUC: {auc_val:.4f}")

# ---------------- Plot: Confusion Matrices ----------------
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

im0 = axes[0].imshow(cm, interpolation='nearest')
axes[0].set_title('Confusion Matrix (Counts)')
axes[0].set_xlabel('Predicted'); axes[0].set_ylabel('True')
axes[0].set_xticks([0,1]); axes[0].set_yticks([0,1])
axes[0].set_xticklabels([idx_to_class[0], idx_to_class[1]])
axes[0].set_yticklabels([idx_to_class[0], idx_to_class[1]])
for i in range(2):
    for j in range(2):
        axes[0].text(j, i, cm[i, j], ha="center", va="center", color="w")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

im1 = axes[1].imshow(cm_norm, vmin=0, vmax=1, cmap='viridis')
axes[1].set_title('Confusion Matrix (Normalized)')
axes[1].set_xlabel('Predicted'); axes[1].set_ylabel('True')
axes[1].set_xticks([0,1]); axes[1].set_yticks([0,1])
axes[1].set_xticklabels([idx_to_class[0], idx_to_class[1]])
axes[1].set_yticklabels([idx_to_class[0], idx_to_class[1]])
for i in range(2):
    for j in range(2):
        axes[1].text(j, i, f"{cm_norm[i, j]:.2f}", ha="center", va="center", color="w")
fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
plt.tight_layout(); plt.show()

# ---------------- Plot: ROC ----------------
fpr, tpr, thr = roc_curve(y_true, y_prob)
plt.figure(figsize=(5,5))
plt.plot(fpr, tpr, label=f"AUC={auc_val:.3f}")
plt.plot([0,1], [0,1], '--')
plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
plt.title("ROC Curve (positive=class 1)")
plt.legend(); plt.grid(True, ls='--', alpha=0.4)
plt.show()


# =========================
# ✅ ROBUST GRAD-CAM HELPERS
# =========================
import numpy as _np
import torch as _torch
from torchcam.methods import SmoothGradCAMpp as _SmoothGradCAMpp
from torchcam.utils import overlay_mask as _overlay_mask

def _extract_single_cam(cams):
    """
    Normalize and return a single 2D numpy array in [0,1].
    Accepts dict/list/tensor/ndarray returned by torchcam.
    """
    # dict -> first value
    if isinstance(cams, dict):
        vals = list(cams.values())
        cams = vals[0] if len(vals)>0 else None

    # list/tuple -> first element
    if isinstance(cams, (list, tuple)):
        cams = cams[0] if len(cams)>0 else None

    if cams is None:
        raise RuntimeError("No CAM produced")

    if _torch.is_tensor(cams):
        arr = cams.detach().cpu().numpy()
    else:
        arr = _np.array(cams)

    arr = _np.squeeze(arr)
    # If more than 2D, try to pick the first channel / collapse dims
    if arr.ndim > 2:
        arr = arr[..., 0]
    if arr.ndim != 2:
        raise RuntimeError(f"CAM extraction failed: expected 2D array, got shape {arr.shape}")

    mn, mx = float(arr.min()), float(arr.max())
    if mx - mn < 1e-8:
        norm = _np.zeros_like(arr, dtype=_np.float32)
    else:
        norm = (arr - mn) / (mx - mn)
    return norm.astype(_np.float32)


def gradcam_on_image_safe(img_path, target_class=None, target_layer="features.7", cam_cls=None):
    """
    Given a path to an image (PIL-safe), return (raw_pil, overlay_pil, probs_np, pred_idx)
    Robust to different CAM return types.
    """
    raw = Image.open(img_path).convert("RGB")
    inp = tfm_val(raw).unsqueeze(0).to(DEVICE)
    inp.requires_grad_(True)

    # ensure model can provide grads/hooks
    was_training = model.training
    model.train()
    # set requires_grad True for params temporarily
    prev_req = [p.requires_grad for p in model.parameters()]
    for p in model.parameters(): p.requires_grad_(True)

    CamCls = cam_cls or _SmoothGradCAMpp
    cam_extractor = CamCls(model, target_layer=target_layer)

    out = model(inp)
    probs = F.softmax(out, dim=1)[0].detach().cpu().numpy()
    pred_idx = int(_np.argmax(probs))
    target_cls = target_class if target_class is not None else pred_idx

    cams = cam_extractor(target_cls, out)
    heatmap = _extract_single_cam(cams)  # 2D float in [0,1]

    # convert to uint8 grayscale PIL (overlay_mask expects 'L' or similar)
    heat_uint8 = (_np.clip(heatmap, 0, 1) * 255.0).round().astype('uint8')
    mask_pil = Image.fromarray(heat_uint8, mode='L')

    overlay = _overlay_mask(raw, mask_pil, alpha=0.5)

    # restore model state
    model.eval() if not was_training else model.train()
    for p, r in zip(model.parameters(), prev_req):
        p.requires_grad_(r)

    return raw, overlay, probs, pred_idx


# ---- visualize a few examples ----
val_paths  = [full_ds.samples[i][0] for i in val_ds.indices]
val_labels = [full_ds.samples[i][1] for i in val_ds.indices]

y_true_np, y_pred_np = y_true, y_pred
correct_idx = [i for i,(p,t) in enumerate(zip(y_pred_np, y_true_np)) if p==t]
wrong_idx   = [i for i,(p,t) in enumerate(zip(y_pred_np, y_true_np)) if p!=t]
show_idxs = (correct_idx[:2] + wrong_idx[:2]) or list(range(min(4, len(val_paths))))

plt.figure(figsize=(12, 3*len(show_idxs)))
for row, i in enumerate(show_idxs, 1):
    img_path = val_paths[i]
    true_lab = idx_to_class[int(y_true_np[i])]
    try:
        raw, overlay, probs, pred_idx = gradcam_on_image_safe(img_path)
    except Exception as e:
        print(f"Grad-CAM failed for {img_path}: {e}")
        # fallback: show raw image only
        raw = Image.open(img_path).convert("RGB")
        overlay = raw
        probs = np.array([0.5, 0.5])
        pred_idx = -1

    pred_lab = idx_to_class[pred_idx] if pred_idx in idx_to_class else f"idx{pred_idx}"
    real_prob = float(probs[1]) if len(probs) > 1 else float(probs[0])

    plt.subplot(len(show_idxs), 2, 2*row-1)
    plt.imshow(raw); plt.axis('off')
    plt.title(f"Original\nTrue: {true_lab}")

    plt.subplot(len(show_idxs), 2, 2*row)
    plt.imshow(overlay); plt.axis('off')
    plt.title(f"Grad-CAM\nPred: {pred_lab} (p[class1]={real_prob:.2f})")

plt.tight_layout(); plt.show()


In [ ]:
# ============================================================
# DFDC → Grad-CAM → On-Chain (Torch 2.8 / EfficientNet / Web3 v6)
# Updated: robust getDetection handling, safer TX logic
# ============================================================

import os, cv2, json, gc, warnings, copy, random, traceback
from glob import glob
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchcam.methods import SmoothGradCAMpp
from torchcam.utils import overlay_mask

from web3 import Web3
from eth_account import Account

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------- CONFIG ----------------
DATA_DIR = os.environ.get("DATA_DIR", "/content/data")
CKPT_PATH = os.environ.get("CKPT_PATH", "/content/best_effnetb0_frames.pth")
IMG_SIZE = int(os.environ.get("IMG_SIZE", 224))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

RPC_URL = os.environ.get("RPC_URL", "https://3927aa499419.ngrok-free.app")  # change to ngrok/alchemy/infura if needed
CONTRACT_ADDRESS = os.environ.get("DEEPFAKE_REG_ADDR", "0x5FbDB2315678afecb367f032d93F642f64180aa3")
PRIVATE_KEY = os.environ.get("HARDHAT_PK", None)  # **set this in Colab env, do NOT hardcode**
CHAIN_ID = int(os.environ.get("CHAIN_ID", 31337))
GAS_LIMIT = int(os.environ.get("GAS_LIMIT", 200_000))

OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", "/content/outputs"))
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# ---------------- CONTRACT ABI ----------------
CONTRACT_ABI = [
    {"anonymous": False, "inputs": [
        {"indexed": True, "internalType": "bytes32", "name": "mediaHash", "type": "bytes32"},
        {"indexed": False, "internalType": "bool", "name": "isFake", "type": "bool"},
        {"indexed": False, "internalType": "uint256", "name": "confidence", "type": "uint256"},
        {"indexed": True, "internalType": "address", "name": "reporter", "type": "address"}],
     "name": "DetectionStored", "type": "event"},
    {"inputs": [{"internalType": "bytes32", "name": "mediaHash", "type": "bytes32"}],
     "name": "getDetection", "outputs": [
        {"internalType": "bool", "name": "exists", "type": "bool"},
        {"internalType": "bool", "name": "isFake", "type": "bool"},
        {"internalType": "uint256", "name": "confidence", "type": "uint256"},
        {"internalType": "address", "name": "reporter", "type": "address"},
        {"internalType": "uint256", "name": "timestamp", "type": "uint256"}],
     "stateMutability": "view", "type": "function"},
    {"inputs": [
        {"internalType": "bytes32", "name": "mediaHash", "type": "bytes32"},
        {"internalType": "bool", "name": "isFake", "type": "bool"},
        {"internalType": "uint256", "name": "confidence", "type": "uint256"}],
     "name": "storeDetection", "outputs": [], "stateMutability": "nonpayable", "type": "function"}
]

# ---------------- MODEL ----------------
tfm_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print("🔄 Loading model...")
model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)

if Path(CKPT_PATH).exists():
    state = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(state["model"])
    print(f"✅ Checkpoint loaded | acc={state.get('acc','N/A')} | epoch={state.get('epoch','N/A')}")
else:
    print(f"⚠️ Checkpoint not found at {CKPT_PATH}, using ImageNet weights only. Predictions will be random.")

model.to(DEVICE).eval()

# ---------------- WEB3 ----------------
w3 = acct = registry = None
if CONTRACT_ADDRESS:
    try:
        w3 = Web3(Web3.HTTPProvider(RPC_URL, request_kwargs={"timeout": 30}))
        if not w3.is_connected():
            raise RuntimeError("❌ Could not connect to RPC_URL. Check your Hardhat/endpoint.")
        if not PRIVATE_KEY:
            print("⚠️ PRIVATE_KEY not set in env. Transactions will fail (read-only checks OK).")
            acct = None
        else:
            acct = Account.from_key(PRIVATE_KEY)
        registry = w3.eth.contract(address=Web3.to_checksum_address(CONTRACT_ADDRESS), abi=CONTRACT_ABI)
        print("✅ Web3 connected:", w3.client_version)
        if acct:
            print("🧑 Reporter:", acct.address)
    except Exception as e:
        print("❌ Web3 setup failed:", e)
        registry = None
else:
    print("⚠️ CONTRACT_ADDRESS not set. On-chain storage will be skipped.")

# ---------------- GRAD-CAM ----------------
def gradcam_on_frame_safe(model, img, layer="features.7"):
    if img is None:
        return None
    torch.set_grad_enabled(True)
    grad_model = copy.deepcopy(model).to(DEVICE)
    grad_model.train()
    grad_model.zero_grad(set_to_none=True)
    for p in grad_model.parameters(): p.requires_grad_(True)

    try:
        inp = tfm_val(img).unsqueeze(0).to(DEVICE)
        inp.requires_grad_(True)
        cam_extractor = SmoothGradCAMpp(grad_model, target_layer=layer)
        with torch.enable_grad():
            out = grad_model(inp)
            pred_idx = int(out.softmax(1).argmax())
            cams = cam_extractor(pred_idx, out)
        heat = cams[0].detach().cpu().numpy().astype(np.float32)
        heat = np.nan_to_num(heat, nan=0.0, posinf=0.0, neginf=0.0)
        # overlay_mask expects a PIL image and a mask image
        mask_img = Image.fromarray((np.clip(heat,0,1)*255).astype("uint8"))
        overlay = overlay_mask(img, mask_img, alpha=0.5)
    except Exception:
        overlay = img
    finally:
        try:
            del grad_model, inp
        except Exception:
            pass
        gc.collect()
        torch.cuda.empty_cache()
    return overlay

# ---------------- HELPERS ----------------
def read_frames(video_path, k=8, size=IMG_SIZE):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return [], None
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    idxs = [max(0,int((i+1)*n/(k+1))-1) for i in range(k)] if n>0 else []
    raw, inp = [], []
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, f = cap.read()
        if not ok or f is None or (hasattr(f, "ndim") and f.ndim !=3) or f.shape[2]<3:
            continue
        f = f[:,:,:3]
        f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        h, w, _ = f.shape
        s = min(h,w)
        y0 = (h-s)//2
        x0 = (w-s)//2
        crop = f[y0:y0+s, x0:x0+s]
        try:
            crop = cv2.resize(crop, (size,size))
        except Exception:
            continue
        pil = Image.fromarray(crop)
        raw.append(pil)
        inp.append(tfm_val(pil))
    cap.release()
    if not inp:
        return [], None
    return raw, torch.stack(inp).to(DEVICE)

def predict_video(video_path):
    raw, x = read_frames(video_path)
    if x is None:
        return None
    if Path(CKPT_PATH).exists():
        with torch.no_grad():
            logits = model(x)
            probs_class1 = F.softmax(logits, dim=1).cpu().numpy()[:,1]  # prob of class index 1
        p_mean = float(np.mean(probs_class1))
        # interpret p_mean as probability of "real" (you can swap depending on training)
        pred = "real" if p_mean > 0.5 else "fake"
    else:
        # Random prediction for testing
        p_mean = random.random()
        pred = "real" if p_mean > 0.5 else "fake"

    # confidence in basis points (0..10000) representing probability * 10000
    conf_bp = int(p_mean * 10000)
    return dict(frames_raw=raw, p_real_mean=p_mean, pred_label=pred, conf_bp=conf_bp, rep_index=max(0,len(raw)//2 - 1))

def keccak_file_hex(p):
    # return 0x... hex of keccak over file bytes
    b = Path(p).read_bytes()
    return Web3.keccak(b).hex()

def safe_get_detection(media_bytes32):
    """
    Call contract.getDetection and return a 5-tuple (exists, isFake, confidence, reporter, timestamp)
    If decoding fails or returns zeros, returns (False, False, 0, '0x0000...', 0)
    """
    if not registry:
        return (False, False, 0, '0x0000000000000000000000000000000000000000', 0)
    try:
        res = registry.functions.getDetection(media_bytes32).call()
        # res should be a tuple. If it decodes but is all zeros, exists==False
        if isinstance(res, (list, tuple)) and len(res) >= 5:
            return res[0], res[1], int(res[2]), res[3], int(res[4])
        else:
            return (False, False, 0, '0x0000000000000000000000000000000000000000', 0)
    except Exception as e:
        # decode error or all-zero return: treat as "not stored"
        return (False, False, 0, '0x0000000000000000000000000000000000000000', 0)

def store_on_chain(media_hash_hex, is_fake, conf_bp):
    """
    Store detection on chain if not already stored.
    Returns:
      - "skipped-no-chain" (no contract)
      - "already" (already present)
      - txhash hex string on success
      - "error:<msg>" on failure
    """
    if not registry:
        return "skipped-no-chain"

    media_bytes32 = Web3.to_bytes(hexstr=media_hash_hex)

    # check existence safely
    exists, _, _, _, _ = safe_get_detection(media_bytes32)
    if exists:
        return "already"

    if acct is None:
        return "skipped-no-acct"

    try:
        nonce = w3.eth.get_transaction_count(acct.address, "pending")
        tx = registry.functions.storeDetection(media_bytes32, bool(is_fake), int(conf_bp)).build_transaction({
            "from": acct.address,
            "nonce": nonce,
            "gas": GAS_LIMIT,
            # EIP-1559 fields if supported; fallback to legacy if not
            "maxFeePerGas": w3.to_wei("1", "gwei"),
            "maxPriorityFeePerGas": w3.to_wei("1", "gwei"),
            "chainId": CHAIN_ID
        })
        signed = acct.sign_transaction(tx)
        tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction)
        receipt = w3.eth.wait_for_transaction_receipt(tx_hash, timeout=120)
        return tx_hash.hex()
    except Exception as e:
        # return a short message but keep full traceback in logs
        print("❗ TX error:", e)
        traceback.print_exc()
        return f"error:{str(e)[:120]}"

# ---------------- MAIN LOOP ----------------
torch.set_grad_enabled(True)
torch.autograd.set_grad_enabled(True)

videos = [p for c in ("real","fake") for p in glob(f"{DATA_DIR}/{c}/*")
          if p.lower().endswith((".mp4",".mov",".avi",".mkv"))]
videos = sorted(videos)[:20]  # deterministic small batch for testing

rows = []
for vp in videos:
    try:
        out = predict_video(vp)
        if out is None:
            print(f"❗ {vp}: No valid frames, skipping.")
            continue

        rep_idx = min(out["rep_index"], len(out["frames_raw"]) - 1)
        sample_frame = out["frames_raw"][rep_idx] if out["frames_raw"] else None
        overlay = gradcam_on_frame_safe(model, sample_frame)
        if overlay is None and sample_frame is not None:
            overlay = sample_frame

        ov_path = OUTPUT_DIR / (Path(vp).stem + "_gradcam.jpg")
        try:
            if overlay is not None:
                overlay.save(ov_path)
        except Exception as e:
            print("⚠️ Could not save overlay:", e)

        mh = keccak_file_hex(vp)
        is_fake = (out["pred_label"] == "fake")
        txh = store_on_chain(mh, is_fake, out["conf_bp"])

        rows.append({
            "video": Path(vp).name,
            "pred_label": out["pred_label"],
            "p_real_mean": round(out["p_real_mean"], 4),
            "confidence_bp": out["conf_bp"],
            "media_hash": mh,
            "tx_status": (txh if txh else "skipped"),
            "overlay_path": str(ov_path) if ov_path.exists() else ""
        })

        print(f"✔ {Path(vp).name} | {out['pred_label']} | conf={out['conf_bp']} | tx={txh[:14] if isinstance(txh, str) else txh}")

    except Exception as e:
        print(f"❗ {vp}: {e}")
        traceback.print_exc()

if len(rows) == 0:
    print("⚠️ No successful predictions, skipping gallery.")
else:
    df = pd.DataFrame(rows)
    display(df)

    try:
        fig, axes = plt.subplots(len(df), 1, figsize=(8, 3 * len(df)))
        if len(df) == 1:
            axes = [axes]
        for ax, (_, r) in zip(axes, df.iterrows()):
            if r["overlay_path"]:
                ax.imshow(Image.open(r["overlay_path"]))
            else:
                ax.text(0.5, 0.5, "no overlay", ha="center", va="center")
            ax.axis("off")
            ax.set_title(f"{r['video']} — {r['pred_label']} | conf={r['confidence_bp']}")
        plt.tight_layout()
        plt.show()
    except Exception:
        print("⚠️ Could not render gallery (headless environment?)")

    out_json = OUTPUT_DIR / "onchain_results.json"
    df.to_json(out_json, orient="records", indent=2)
    print(f"\n📦 Results → {out_json}\n🔗 RPC={RPC_URL}")


In [ ]:
# ============================================================
# DFDC → Grad-CAM → On-Chain (Torch 2.8 / EfficientNet / Web3 v6)
# ============================================================

import os, cv2, json, gc, warnings, copy, random
from glob import glob
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch, torch.nn as nn, torch.nn.functional as F
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchcam.methods import SmoothGradCAMpp
from torchcam.utils import overlay_mask

from web3 import Web3
from eth_account import Account

warnings.filterwarnings("ignore", category=FutureWarning)

# ---------------- CONFIG ----------------
DATA_DIR = "/content/data"
CKPT_PATH = "/content/best_effnetb0_frames.pth"  # DFDC-trained checkpoint
IMG_SIZE = 224
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

RPC_URL = os.environ.get("RPC_URL","https://d0f8512f8462.ngrok-free.app")
CONTRACT_ADDRESS = os.environ.get("DEEPFAKE_REG_ADDR", "0x5FbDB2315678afecb367f032d93F642f64180aa3")
PRIVATE_KEY = os.environ.get("HARDHAT_PK", "0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80")
CHAIN_ID = 31337
GAS_LIMIT = 200_000

OUTPUT_DIR = Path("/content/outputs")
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

# ---------------- CONTRACT ABI ----------------
CONTRACT_ABI = [
    {"anonymous": False, "inputs": [
        {"indexed": True, "internalType": "bytes32", "name": "mediaHash", "type": "bytes32"},
        {"indexed": False, "internalType": "bool", "name": "isFake", "type": "bool"},
        {"indexed": False, "internalType": "uint256", "name": "confidence", "type": "uint256"},
        {"indexed": True, "internalType": "address", "name": "reporter", "type": "address"}],
     "name": "DetectionStored", "type": "event"},
    {"inputs": [{"internalType": "bytes32", "name": "mediaHash", "type": "bytes32"}],
     "name": "getDetection", "outputs": [
        {"internalType": "bool", "name": "exists", "type": "bool"},
        {"internalType": "bool", "name": "isFake", "type": "bool"},
        {"internalType": "uint256", "name": "confidence", "type": "uint256"},
        {"internalType": "address", "name": "reporter", "type": "address"},
        {"internalType": "uint256", "name": "timestamp", "type": "uint256"}],
     "stateMutability": "view", "type": "function"},
    {"inputs": [
        {"internalType": "bytes32", "name": "mediaHash", "type": "bytes32"},
        {"internalType": "bool", "name": "isFake", "type": "bool"},
        {"internalType": "uint256", "name": "confidence", "type": "uint256"}],
     "name": "storeDetection", "outputs": [], "stateMutability": "nonpayable", "type": "function"}
]

# ---------------- MODEL ----------------
tfm_val = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

print("🔄 Loading model...")
model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 2)

if Path(CKPT_PATH).exists():
    state = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(state["model"])
    print(f"✅ Checkpoint loaded | acc={state.get('acc','N/A')} | epoch={state.get('epoch','N/A')}")
else:
    print(f"⚠️ Checkpoint not found at {CKPT_PATH}, using ImageNet weights only. Predictions will be random.")

model.to(DEVICE).eval()

# ---------------- WEB3 ----------------
if CONTRACT_ADDRESS:
    w3 = Web3(Web3.HTTPProvider(RPC_URL, request_kwargs={"timeout": 30}))
    if not w3.is_connected():
        raise RuntimeError("❌ Could not connect to RPC_URL. Check your Hardhat/endpoint.")
    acct = Account.from_key(PRIVATE_KEY)
    registry = w3.eth.contract(address=Web3.to_checksum_address(CONTRACT_ADDRESS), abi=CONTRACT_ABI)
    print("✅ Web3 connected:", w3.client_version)
    print("🧑 Reporter:", acct.address)
else:
    print("⚠️ CONTRACT_ADDRESS not set. On-chain storage will be skipped.")
    w3 = acct = registry = None

# ---------------- GRAD-CAM ----------------
def gradcam_on_frame_safe(model, img, layer="features.7"):
    if img is None:
        return None
    torch.set_grad_enabled(True)
    grad_model = copy.deepcopy(model).to(DEVICE)
    grad_model.train()
    grad_model.zero_grad(set_to_none=True)
    for p in grad_model.parameters(): p.requires_grad_(True)

    try:
        inp = tfm_val(img).unsqueeze(0).to(DEVICE)
        inp.requires_grad_(True)
        cam_extractor = SmoothGradCAMpp(grad_model, target_layer=layer)
        with torch.enable_grad():
            out = grad_model(inp)
            pred_idx = int(out.softmax(1).argmax())
            cams = cam_extractor(pred_idx, out)
        heat = cams[0].detach().cpu().numpy().astype(np.float32)
        heat = np.nan_to_num(heat, nan=0.0, posinf=0.0, neginf=0.0)
        overlay = overlay_mask(img, Image.fromarray((np.clip(heat,0,1)*255).astype("uint8")), alpha=0.5)
    except Exception:
        overlay = img
    del grad_model, inp
    gc.collect()
    torch.cuda.empty_cache()
    return overlay

# ---------------- HELPERS ----------------
def read_frames(video_path, k=8, size=IMG_SIZE):
    cap = cv2.VideoCapture(video_path)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    idxs = [max(0,int((i+1)*n/(k+1))-1) for i in range(k)] if n>0 else []
    raw, inp = [], []
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, f = cap.read()
        if not ok or f is None or f.ndim !=3 or f.shape[2]<3:
            continue
        f = f[:,:,:3]
        f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        h, w, _ = f.shape
        s = min(h,w)
        y0 = (h-s)//2
        x0 = (w-s)//2
        f = cv2.resize(f[y0:y0+s, x0:x0+s], (size,size))
        pil = Image.fromarray(f)
        raw.append(pil)
        inp.append(tfm_val(pil))
    cap.release()
    if not inp:
        return [], None
    return raw, torch.stack(inp).to(DEVICE)

def predict_video(video_path):
    raw, x = read_frames(video_path)
    if x is None:
        return None
    if Path(CKPT_PATH).exists():
        with torch.no_grad():
            logits = model(x)
            probs = F.softmax(logits, dim=1).cpu().numpy()[:,1]
        p_mean = float(np.mean(probs))
        pred = "fake" if p_mean < 0.5 else "real"
    else:
        # Random prediction for testing
        p_mean = random.random()
        pred = "fake" if p_mean < 0.5 else "real"

    conf = int(abs(p_mean-0.5)*2*10000)
    return dict(frames_raw=raw, p_real_mean=p_mean, pred_label=pred, conf_bp=conf, rep_index=len(raw)//2)

def keccak_file_hex(p):
    return Web3.keccak(Path(p).read_bytes()).hex()

def store_on_chain(media_hash_hex, is_fake, conf_bp):
    if not registry:
        return "skipped-no-chain"
    media_bytes32 = Web3.to_bytes(hexstr=media_hash_hex)
    exists, *_ = registry.functions.getDetection(media_bytes32).call()
    if exists:
        return "already"
    nonce = w3.eth.get_transaction_count(acct.address, "pending")
    tx = registry.functions.storeDetection(media_bytes32, bool(is_fake), int(conf_bp)).build_transaction({
        "from": acct.address, "nonce": nonce, "gas": GAS_LIMIT,
        "maxFeePerGas": w3.to_wei("1", "gwei"),
        "maxPriorityFeePerGas": w3.to_wei("1", "gwei"),
        "chainId": CHAIN_ID
    })
    signed = acct.sign_transaction(tx)
    tx_hash = w3.eth.send_raw_transaction(signed.raw_transaction)
    w3.eth.wait_for_transaction_receipt(tx_hash)
    return tx_hash.hex()

# ---------------- MAIN LOOP ----------------
torch.set_grad_enabled(True)
torch.autograd.set_grad_enabled(True)

videos = [p for c in ("real","fake") for p in glob(f"{DATA_DIR}/{c}/*")
          if p.lower().endswith((".mp4",".mov",".avi",".mkv"))]
videos = videos[:20]  # limit for testing

rows = []
for vp in videos:
    try:
        out = predict_video(vp)
        if out is None:
            print(f"❗ {vp}: No valid frames, skipping.")
            continue
        overlay = gradcam_on_frame_safe(model, out["frames_raw"][out["rep_index"]])
        ov_path = OUTPUT_DIR / (Path(vp).stem+"_gradcam.jpg")
        overlay.save(ov_path)
        mh = keccak_file_hex(vp)
        is_fake = out["pred_label"]=="fake"
        txh = store_on_chain(mh, is_fake, out["conf_bp"])
        rows.append({
            "video": Path(vp).name,
            "pred_label": out["pred_label"],
            "p_real_mean": round(out["p_real_mean"],4),
            "confidence_bp": out["conf_bp"],
            "media_hash": mh,
            "tx_status": txh[:14]+"..." if txh else "skipped",
            "overlay_path": str(ov_path)
        })
        print(f"✔ {Path(vp).name} | {out['pred_label']} | conf={out['conf_bp']} | tx={txh[:14] if txh else 'skipped'}...")
    except Exception as e:
        print(f"❗ {vp}: {e}")

if len(rows)==0:
    print("⚠️ No successful predictions, skipping gallery.")
else:
    df = pd.DataFrame(rows)
    display(df)

    fig, axes = plt.subplots(len(df),1,figsize=(8,3*len(df)))
    if len(df)==1: axes=[axes]
    for ax,(_,r) in zip(axes, df.iterrows()):
        ax.imshow(Image.open(r["overlay_path"]))
        ax.axis("off")
        ax.set_title(f"{r['video']} — {r['pred_label']} | conf={r['confidence_bp']}")
    plt.tight_layout()
    plt.show()

    out_json = OUTPUT_DIR / "onchain_results.json"
    df.to_json(out_json, orient="records", indent=2)
    print(f"\n📦 Results → {out_json}\n🔗 RPC={RPC_URL}")


future extension "" streamlit ""


In [ ]:

%pip install -q --upgrade pip

# Install streamlit & pyngrok but skip dependency resolution so numpy stays as-is
%pip install -q --no-deps streamlit pyngrok


In [ ]:
import streamlit, torch, numpy, pyngrok
print("streamlit:", streamlit.__version__)
print("torch:", torch.__version__)
print("numpy:", numpy.__version__)


In [ ]:
%%bash
set -euo pipefail
echo "=== CLEAN broken ~inja2 distributions (if present) ==="
# remove obvious broken package files left from failed installs (non-destructive best-effort)
# NOTE: this uses sudo-free removal from site-packages — safe in Colab ephemeral environment
PYPKG_DIR=$(python -c "import site; print(site.getsitepackages()[0])")
echo "site-packages:", "$PYPKG_DIR"
# remove weird files starting with ~inja2 or ~markupsafe
rm -f "${PYPKG_DIR}"/~inja2* "${PYPKG_DIR}"/~markupsafe* || true
rm -f "${PYPKG_DIR}"/.~inja2* "${PYPKG_DIR}"/.~markupsafe* || true

echo "=== Reinstall Jinja2 & MarkupSafe cleanly ==="
pip install -q --upgrade --force-reinstall "Jinja2==3.1.2" "MarkupSafe==2.1.3"

echo "=== Install Streamlit & pyngrok (no-deps to avoid numpy churn) ==="
pip install -q --no-deps streamlit pyngrok

echo "=== Install PyTorch (cu118 wheel for Colab GPU) ==="
pip install -q --upgrade --force-reinstall "torch>=2.2.0" "torchvision>=0.15.0" --index-url https://download.pytorch.org/whl/cu118

echo "=== Install other helpers without forcing numpy ==="
pip install -q --no-deps pillow opencv-python-headless torchcam web3 eth-account pandas scikit-learn tqdm

echo "=== Write Streamlit app to /content/streamlit_dfdc_gradcam_onchain.py ==="
cat > /content/streamlit_dfdc_gradcam_onchain.py <<'PY'
# streamlit_dfdc_gradcam_onchain.py
# DFDC frames -> EfficientNet-B0 -> Grad-CAM -> optional on-chain (Colab-friendly)

import os, io, gc, tempfile, time, traceback
from pathlib import Path
from typing import List, Optional

import streamlit as st
import pandas as pd
import numpy as np
from PIL import Image
import cv2

import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import transforms
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torchcam.methods import SmoothGradCAMpp
from torchcam.utils import overlay_mask

# Optional web3
try:
    from web3 import Web3
    from eth_account import Account
    WEB3_AVAILABLE = True
except Exception:
    WEB3_AVAILABLE = False

# Config
DATA_DIR = os.environ.get("DATA_DIR", "/content/data")
CKPT_PATH = os.environ.get("CKPT_PATH", "/content/best_effnetb0_frames.pth")
IMG_SIZE = int(os.environ.get("IMG_SIZE", 224))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
TARGET_LAYER = "features.7"
K_DEFAULT = 8

CONTRACT_ABI = [
    {"inputs":[{"internalType":"bytes32","name":"mediaHash","type":"bytes32"}],"name":"getDetection","outputs":[{"internalType":"bool","name":"exists","type":"bool"},{"internalType":"bool","name":"isFake","type":"bool"},{"internalType":"uint256","name":"confidence","type":"uint256"},{"internalType":"address","name":"reporter","type":"address"},{"internalType":"uint256","name":"timestamp","type":"uint256"}],"stateMutability":"view","type":"function"},
    {"inputs":[{"internalType":"bytes32","name":"mediaHash","type":"bytes32"},{"internalType":"bool","name":"isFake","type":"bool"},{"internalType":"uint256","name":"confidence","type":"uint256"}],"name":"storeDetection","outputs":[],"stateMutability":"nonpayable","type":"function"}
]

st.set_page_config(page_title="DFDC Grad-CAM OnChain", layout="wide")
st.title("DFDC → Grad-CAM → (optional) On-Chain registry")
st.write("Upload or pick a video. The app extracts K frames, runs inference, shows Grad-CAM overlay and can store a detection on-chain.")

@st.cache_resource(show_spinner=False)
def load_model_and_tfm(ckpt_path: str = CKPT_PATH, device: str = DEVICE):
    m = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
    m.classifier[1] = nn.Linear(m.classifier[1].in_features, 2)
    if os.path.exists(ckpt_path):
        try:
            state = torch.load(ckpt_path, map_location=device)
            m.load_state_dict(state["model"])
        except Exception as e:
            st.warning(f"Couldn't load checkpoint at {ckpt_path}: {e}")
    m.to(device)
    m.eval()
    tfm = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.ToTensor(),
        transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
    ])
    return m, tfm

model, tfm_val = load_model_and_tfm()

def extract_k_frames_from_path(video_path: str, k: int = K_DEFAULT, size: int = IMG_SIZE) -> List[Image.Image]:
    cap = cv2.VideoCapture(str(video_path))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 0
    if n <= 0:
        cap.release()
        return []
    idxs = [max(0, int((i + 1) * n / (k + 1)) - 1) for i in range(k)]
    raws: List[Image.Image] = []
    for idx in idxs:
        cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
        ok, f = cap.read()
        if not ok or f is None:
            continue
        if f.ndim != 3 or f.shape[2] < 3:
            continue
        f = f[:, :, :3]
        f = cv2.cvtColor(f, cv2.COLOR_BGR2RGB)
        h, w, _ = f.shape
        s = min(h, w)
        y0 = (h - s) // 2
        x0 = (w - s) // 2
        crop = cv2.resize(f[y0:y0+s, x0:x0+s], (size, size))
        raws.append(Image.fromarray(crop))
    cap.release()
    return raws

def predict_from_frames(frames: List[Image.Image], device: str = DEVICE):
    if not frames:
        return None
    xs = torch.stack([tfm_val(img) for img in frames]).to(device)
    with torch.no_grad():
        logits = model(xs)
        probs = F.softmax(logits, dim=1).cpu().numpy()[:, 1]
    p_mean = float(np.mean(probs))
    pred_label = "real" if p_mean >= 0.5 else "fake"
    conf_bp = int(abs(p_mean - 0.5) * 2 * 10000)
    rep_index = len(frames) // 2
    return dict(p_real_mean=p_mean, pred_label=pred_label, conf_bp=conf_bp, rep_index=rep_index, probs=probs)

def safe_gradcam_overlay(img_pil: Image.Image, target_layer: str = TARGET_LAYER, timeout_s: float = 12.0) -> Image.Image:
    start = time.time()
    try:
        gm = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)
        gm.classifier[1] = nn.Linear(gm.classifier[1].in_features, 2)
        gm.load_state_dict(model.state_dict())
        gm.to(DEVICE).train()
        for p in gm.parameters():
            p.requires_grad_(True)
        inp = tfm_val(img_pil).unsqueeze(0).to(DEVICE)
        inp.requires_grad_(True)
        cam = SmoothGradCAMpp(gm, target_layer=target_layer)
        with torch.enable_grad():
            out = gm(inp)
            pred_idx = int(out.softmax(1).argmax())
            cams = cam(pred_idx, out)
        heat = cams[0].detach().cpu().numpy()
        heat = np.nan_to_num(heat, nan=0.0, posinf=0.0, neginf=0.0)
        heat_uint8 = (np.clip(heat, 0, 1) * 255.0).astype('uint8')
        mask_pil = Image.fromarray(heat_uint8, mode='L')
        overlay = overlay_mask(img_pil, mask_pil, alpha=0.5)
    except Exception:
        overlay = img_pil
    finally:
        try:
            del gm, inp, out, cams
        except Exception:
            pass
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    if time.time() - start > timeout_s:
        return img_pil
    return overlay

def keccak_file_hex(path: str) -> Optional[str]:
    try:
        return Web3.keccak(Path(path).read_bytes()).hex()
    except Exception:
        return None

def connect_web3(rpc_url: str, private_key: str, contract_addr: str):
    if not WEB3_AVAILABLE:
        raise RuntimeError("web3/eth-account not installed in environment")
    w3 = Web3(Web3.HTTPProvider(rpc_url, request_kwargs={"timeout": 30}))
    if not w3.is_connected():
        raise RuntimeError("Could not connect to RPC URL")
    acct = Account.from_key(private_key)
    ctr = w3.eth.contract(address=Web3.to_checksum_address(contract_addr), abi=CONTRACT_ABI)
    return w3, acct, ctr

st.sidebar.header("Configuration")
st.sidebar.write(f"Device: **{DEVICE}** | CKPT: **{CKPT_PATH}**")

video_files = []
for c in ("real", "fake"):
    p = Path(DATA_DIR) / c
    if p.exists():
        for f in sorted(p.glob("*")):
            if f.suffix.lower() in [".mp4", ".mov", ".avi", ".mkv"]:
                video_files.append(str(f))

uploaded = st.sidebar.file_uploader("Or upload a video", type=["mp4","mov","avi","mkv"])
sel = st.sidebar.selectbox("Pick a video from DATA_DIR (optional)", [""] + video_files)
K_FRAMES = st.sidebar.number_input("K frames per video", value=K_DEFAULT, min_value=1, max_value=32)

st.sidebar.markdown("---")
st.sidebar.header("On-Chain (optional)")
use_chain = st.sidebar.checkbox("Enable on-chain storage", value=False)
rpc_url = st.sidebar.text_input("RPC URL", value=os.environ.get("RPC_URL", "") if use_chain else "")
contract_addr = st.sidebar.text_input("Contract address", value=os.environ.get("DEEPFAKE_REG_ADDR", "") if use_chain else "")
private_key = st.sidebar.text_input("Private key (hex)", type="password", value=os.environ.get("HARDHAT_PK", "") if use_chain else "")

col1, col2 = st.columns([1, 2])
with col1:
    st.subheader("Input")
    if uploaded is not None:
        tfile = tempfile.NamedTemporaryFile(delete=False, suffix=Path(uploaded.name).suffix)
        tfile.write(uploaded.getbuffer())
        tfile.flush()
        video_path = tfile.name
        st.video(uploaded)
    elif sel:
        video_path = sel
        try:
            with open(video_path, "rb") as vf:
                st.video(vf.read())
        except Exception:
            st.write("Selected video preview not available")
    else:
        st.info("Select a video from the sidebar or upload a video.")
        video_path = None

    run_button = st.button("Run inference")

with col2:
    st.subheader("Results")
    results_placeholder = st.empty()

if run_button and video_path:
    try:
        st.info("Extracting frames...")
        frames = extract_k_frames_from_path(video_path, k=K_FRAMES, size=IMG_SIZE)
        if not frames:
            st.error("No usable frames extracted from the video.")
        else:
            st.success(f"Extracted {len(frames)} frames — running model...")
            res = predict_from_frames(frames)
            rep_idx = res["rep_index"]
            rep_img = frames[rep_idx]
            st.info("Generating Grad-CAM overlay (may take a few seconds)...")
            overlay = safe_gradcam_overlay(rep_img)

            c1, c2 = st.columns(2)
            c1.image(rep_img, caption="Representative frame", use_column_width=True)
            c2.image(overlay, caption="Grad-CAM overlay", use_column_width=True)

            df_row = {
                "video": Path(video_path).name,
                "pred_label": res["pred_label"],
                "p_real_mean": round(res["p_real_mean"], 4),
                "confidence_bp": res["conf_bp"],
            }
            df = pd.DataFrame([df_row])
            results_placeholder.table(df)

            if st.button("Save overlay to outputs folder"):
                outdir = Path("outputs")
                outdir.mkdir(exist_ok=True)
                out_path = outdir / (Path(video_path).stem + "_gradcam.jpg")
                overlay.save(out_path)
                st.success(f"Overlay saved to {out_path}")

            if use_chain:
                if not WEB3_AVAILABLE:
                    st.error("web3/eth-account not installed — cannot store on-chain")
                else:
                    if st.button("Store detection on-chain"):
                        try:
                            w3, acct, ctr = connect_web3(rpc_url, private_key, contract_addr)
                            media_hash = keccak_file_hex(video_path)
                            if media_hash is None:
                                st.error("Failed to compute keccak of file bytes.")
                            else:
                                media_bytes32 = Web3.to_bytes(hexstr=media_hash)
                                exists, *_ = ctr.functions.getDetection(media_bytes32).call()
                                if exists:
                                    st.warning("This media is already stored on-chain.")
                                else:
                                    nonce = w3.eth.get_transaction_count(acct.address, "pending")
                                    tx = ctr.functions.storeDetection(media_bytes32, res["pred_label"]=="fake", int(res["conf_bp"]) ).build_transaction({
                                        "from": acct.address, "nonce": nonce, "gas": 200000,
                                        "maxFeePerGas": w3.to_wei("1","gwei"),
                                        "maxPriorityFeePerGas": w3.to_wei("1","gwei"),
                                        "chainId": w3.eth.chain_id
                                    })
                                    signed = acct.sign_transaction(tx)
                                    txh = w3.eth.send_raw_transaction(signed.raw_transaction)
                                    w3.eth.wait_for_transaction_receipt(txh)
                                    st.success(f"Stored on-chain — tx {txh.hex()}")
                        except Exception as e:
                            st.error(f"On-chain error: {e}\\n{traceback.format_exc()}")

    except Exception as e:
        st.error(f"Error during inference: {e}\\n{traceback.format_exc()}")

else:
    if run_button and not video_path:
        st.error("No video selected or uploaded.")

st.caption("App adapted from your Colab DFDC -> Grad-CAM -> Web3 notebook. Adjust DATA_DIR / CKPT_PATH as needed.")
PY

echo "=== start streamlit ==="
fuser -k 8501/tcp || true
nohup streamlit run /content/streamlit_dfdc_gradcam_onchain.py --server.port 8501 --server.headless true > /content/streamlit.log 2>&1 &

sleep 3

# If NGROK auth token exists in env, use it. Otherwise do not try to create a tunnel.
if [ -n "${NGROK_AUTH_TOKEN:-}" ]; then
  echo "=== NGROK auth token found — starting tunnel ==="
  python3 - <<'PY'
from pyngrok import ngrok
import os
from pyngrok import conf
conf.get_default().auth_token = os.environ.get("NGROK_AUTH_TOKEN")
t = ngrok.connect(8501, "http")
print("Streamlit app is live at:", t.public_url)
PY
else
  echo "=== NGROK_AUTH_TOKEN not set. To get a public URL, set an ngrok authtoken and re-run the tunnel step:"
  echo "1) Sign up at https://dashboard.ngrok.com/signup"
  echo "2) Get your authtoken and in Colab run:"
  echo "   import os; os.environ['NGROK_AUTH_TOKEN']='your_token_here'"
  echo "   # then run the tunnel with:"
  echo "   python3 - <<'PY'"
  echo "from pyngrok import ngrok, conf; import os; conf.get_default().auth_token = os.environ['NGROK_AUTH_TOKEN']; print(ngrok.connect(8501,'http').public_url)"
  echo "PY"
fi

echo "=== Streamlit log (first 80 lines) ==="
sleep 2
head -n 80 /content/streamlit.log || true


In [ ]:
# -------------------------------
# 1️⃣ Clean broken Jinja2
# -------------------------------
!rm -rf /usr/local/lib/python3.12/dist-packages/~inja2*
!pip install --upgrade --force-reinstall Jinja2 MarkupSafe

# -------------------------------
# 2️⃣ Install required packages
# -------------------------------
!pip install --upgrade streamlit pyngrok torch torchvision torchaudio opencv-python-headless

# -------------------------------
# 3️⃣ Set your ngrok auth token
# -------------------------------
import os
from pyngrok import ngrok, conf

NGROK_TOKEN = "346uBHg2QvcgIAjGoALvzaR4NkO_3ALPkgQXHHJ46hekRukrd"  # <- your new valid token
os.environ['NGROK_AUTH_TOKEN'] = NGROK_TOKEN
conf.get_default().auth_token = NGROK_TOKEN

# -------------------------------
# 4️⃣ Write Streamlit app (example)
# -------------------------------
app_code = """
import streamlit as st
st.title('Deepfake Detection Demo')
st.write('Hello! This is your Streamlit app running in Colab with pyngrok.')
"""

with open("/content/streamlit_dfdc_gradcam_onchain.py", "w") as f:
    f.write(app_code)

# -------------------------------
# 5️⃣ Start Streamlit in background
# -------------------------------
get_ipython().system_raw("streamlit run /content/streamlit_dfdc_gradcam_onchain.py &")

# -------------------------------
# 6️⃣ Start ngrok tunnel and show public URL
# -------------------------------
public_url = ngrok.connect(8501, "http").public_url
print("Your Streamlit public URL:", public_url)
